In [ ]:
import os
import logging
from dotenv import load_dotenv
from logging.handlers import RotatingFileHandler
import json
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
#from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import nest_asyncio
from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings

In [ ]:
# 환경 변수 로드
load_dotenv()
#print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("UPSTAGE_API_KEY"))

# 로거 설정
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
logger.addHandler(console_handler)

## 요약 3개 생성하기

In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from dotenv import load_dotenv

# 1. API 키 로드
load_dotenv()

client = ChatUpstage(
    model="solar-pro",
    temperature=0.5
)

# ---------------------------------------------------------
# [함수] 3가지 버전의 요약 생성
# ---------------------------------------------------------
def generate_three_summaries(original_answer, question):
    summaries = []
    
    system_message = """
    You are a wise and compassionate **Christian** counselor chatbot. 
    Your goal is to share the love of Jesus through gentle, conversational interactions.
    
    **Instructions:**
    1.  **Summarize:** The user provides a 'Question' and a 'Long Theological Answer'. You must rewrite the answer into a concise, conversational response.
    2.  **Tone:** Speak naturally like a caring friend or pastor, not like a textbook or a search engine. Be empathetic and warm.
    3.  **Length:** Keep it relatively short (2-4 sentences usually, unless the topic requires more nuance), suitable for a chat interface.
    4.  **Content:** Base your answer STRICTLY on the provided 'Long Theological Answer'. Do not invent new theology, but you can phrase it more simply.
    5.  **Style:** Avoid heavy theological jargon where possible. If the topic is sensitive, show understanding.
    """
    
    for i in range(3):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": f"Question: {question}\n\nOriginal Answer: {original_answer}\n\nSummarized Answer {i+1}:"}
                ],
                temperature=0.7 + (i * 0.1) 
            )
            summaries.append(response.choices[0].message.content.strip())
        except Exception as e:
            # 에러 발생 시 빈 문자열 대신 에러 메시지를 남기거나 비워둡니다.
            print(f"⚠️ Error generating summary {i+1}: {e}")
            summaries.append("") # 실패하면 빈칸 저장
    
    return summaries

# ---------------------------------------------------------
# [실행] 데이터셋에 적용 (중간 저장 기능 포함)
# ---------------------------------------------------------
input_file = "GotQuestions_raw_Ara_2025-05-01.xlsx"
df = pd.read_excel(input_file)

# 결과를 담을 리스트
summary_results = []

# 최종 저장 파일명
output_file = "GQ_3answers_2175.xlsx"
# 중간 백업 파일명
backup_file = "3Ans_GQ_backup.xlsx"

print(f"🚀 총 {len(df)}개 데이터에 대해 요약 생성 시작...")

# enumerate를 사용하여 몇 번째인지 카운트
for i, (index, row) in enumerate(tqdm(df.iterrows(), total=df.shape[0])):
    try:
        question = row['Question_ENG']
        original = row['Answer_ENG']
        
        # 3가지 요약 생성
        three_summaries = generate_three_summaries(original, question)
        
        summary_results.append({
            "Question": question,
            "Original_Answer": original,
            "Summary1": three_summaries[0],
            "Summary2": three_summaries[1],
            "Summary3": three_summaries[2]
        })
        
        # ★안전장치★: 10개 할 때마다 파일로 저장 (백업)
        # 컴퓨터가 꺼지거나 API 에러로 멈춰도 이 파일은 남음
        if (i + 1) % 10 == 0:
            pd.DataFrame(summary_results).to_excel(backup_file, index=False)
            
    except Exception as e:
        print(f"❌ Row {i} Error: {e}")
        # 행 전체 에러가 나도 멈추지 않고 다음 행으로 넘어가기
        continue

# 반복문이 다 끝나면 최종 파일 저장
result_df = pd.DataFrame(summary_results)
result_df.to_excel(output_file, index=False)

print(f"✅ 모든 작업 완료! 최종 파일: {output_file}")

## Ragas로 Bast 답변 계산

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
import nest_asyncio
import time

# 1. 환경 설정 및 모델 준비
nest_asyncio.apply()

model = ChatUpstage(model="solar-pro", temperature=0.8)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

#model = ChatOpenAI(model="gpt-4o")
#embeddings = OpenAIEmbeddings()

# 2. 파일 로드
file_summary = "GQ_3answers_2175.xlsx"
file_original = "GotQuestions_raw_Ara_2025-05-01.xlsx"
output_file = "[Ver2]GQ_Ragas_Progress_upstage.xlsx" # 중간 저장 파일명

print(f"📂 파일 로딩 중...")
df_summary = pd.read_excel(file_summary)
df_original = pd.read_excel(file_original)

# 3. 데이터 병합 (이전과 동일 로직)
def get_question_col(df):
    for col in ['Question', 'Question_ENG']:
        if col in df.columns:
            return col
    return df.columns[0]

q_col_sum = get_question_col(df_summary)
q_col_orig = get_question_col(df_original)

df_merged = pd.merge(
    df_summary, 
    df_original[[q_col_orig, 'Answer_ENG']], 
    left_on=q_col_sum, 
    right_on=q_col_orig, 
    how='left'
)

ground_truth_col = 'Answer_ENG'
final_q_col = q_col_sum
summary_cols = ['Summary1', 'Summary2', 'Summary3']

# 4. 배치 처리 설정
BATCH_SIZE = 10  # 10개씩 처리
total_rows = len(df_merged)
processed_results = [] # 결과를 모을 리스트

print(f"🚀 총 {total_rows}개의 데이터를 {BATCH_SIZE}개씩 나누어 평가합니다.")

# 가중치 설정
W_REL, W_COR, W_SIM = 0.5, 0.22, 0.28

def pick_best_weighted(row):
    """행별로 점수를 계산해 Best를 뽑는 함수"""
    scores = []
    answers = []
    
    for i in range(1, 4):
        # 점수가 계산되지 않았거나 에러인 경우 0점 처리
        s_rel = row.get(f'Summary{i}_Relevancy', 0)
        s_cor = row.get(f'Summary{i}_Correctness', 0)
        s_sim = row.get(f'Summary{i}_Similarity', 0)
        
        # NaN 체크
        if pd.isna(s_rel): s_rel = 0
        if pd.isna(s_cor): s_cor = 0
        if pd.isna(s_sim): s_sim = 0
        
        weighted_score = (s_rel * W_REL) + (s_cor * W_COR) + (s_sim * W_SIM)
        scores.append(weighted_score)
        answers.append(row.get(f'Summary{i}', ""))
    
    max_score = max(scores)
    max_index = scores.index(max_score)
    
    return pd.Series({
        'Best_Summary': answers[max_index], 
        'Weighted_Score': max_score, 
        'Best_Source_Num': f"Summary {max_index+1}"
    })

# 5. 메인 루프 (배치 단위 실행)
for start_idx in range(0, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    print(f"\n🔄 Processing Batch: {start_idx} ~ {end_idx} (Total: {total_rows})")
    
    # 현재 배치 데이터 슬라이싱
    batch_df = df_merged.iloc[start_idx:end_idx].copy()
    
    # 필수 데이터 리스트 변환 (NaN 처리 포함)
    questions = batch_df[final_q_col].fillna("").astype(str).tolist()
    ground_truths = batch_df[ground_truth_col].fillna("").astype(str).tolist()
    contexts = [[gt] for gt in ground_truths]
    
    # 현재 배치의 결과를 담을 임시 데이터프레임 (원본 복사)
    current_batch_result = batch_df.copy()
    
    try:
        # Summary 1, 2, 3 각각 평가
        for i, col in enumerate(summary_cols):
            num = i + 1
            answers = batch_df[col].fillna("").astype(str).tolist()
            
            # Ragas 데이터셋 생성
            data_dict = {
                "question": questions,
                "ground_truth": ground_truths,
                "answer": answers,
                "contexts": contexts
            }
            dataset = Dataset.from_dict(data_dict)
            
            print(f"   ...Evaluating {col} ({len(dataset)} items)")
            
            # 평가 수행
            results = evaluate(
                dataset=dataset,
                metrics=[answer_correctness, answer_similarity, answer_relevancy],
                llm=model,
                embeddings=embeddings,
                raise_exceptions=False # 에러 발생 시 멈추지 않고 NaN 반환 시도
            )
            
            # 결과 DataFrame으로 변환
            res_df = results.to_pandas()
            
            # 결과를 현재 배치 DataFrame에 컬럼으로 추가
            current_batch_result[f'Summary{num}_Correctness'] = res_df['answer_correctness'].values
            current_batch_result[f'Summary{num}_Similarity'] = res_df['answer_similarity'].values
            current_batch_result[f'Summary{num}_Relevancy'] = res_df['answer_relevancy'].values

        # Best Pick 계산
        best_pick_df = current_batch_result.apply(pick_best_weighted, axis=1)
        current_batch_result = pd.concat([current_batch_result, best_pick_df], axis=1)
        
        # 결과 리스트에 추가
        processed_results.append(current_batch_result)
        
        # --- 중간 저장 ----
        # 지금까지 처리된 모든 결과 합치기
        all_results_so_far = pd.concat(processed_results, axis=0)
        
        # 컬럼 순서 정리 (보기 좋게)
        cols_order = [final_q_col, ground_truth_col] + summary_cols
        metrics_cols = []
        for i in range(1, 4):
            metrics_cols.extend([f'Summary{i}_Correctness', f'Summary{i}_Similarity', f'Summary{i}_Relevancy'])
        final_cols = cols_order + metrics_cols + ['Best_Source_Num', 'Weighted_Score', 'Best_Summary']
        
        # 존재하는 컬럼만 선택하여 저장
        existing_cols = [c for c in final_cols if c in all_results_so_far.columns]
        # 나머지 기타 컬럼들 뒤에 붙이기
        remaining_cols = [c for c in all_results_so_far.columns if c not in existing_cols]
        
        save_df = all_results_so_far[existing_cols + remaining_cols]
        save_df.to_excel(output_file, index=False)
        
        print(f"   💾 중간 저장 완료: {end_idx}행까지 저장됨 -> {output_file}")

    except Exception as e:
        print(f"❌ [CRITICAL ERROR] Batch {start_idx}~{end_idx} 처리 중 오류 발생!")
        print(f"Error Message: {e}")
        print("🚨 현재까지 저장된 파일을 유지하고 멈춥니다. API 키를 확인하거나 나중에 다시 시도하세요.")
        break # 루프 중단

print("\n✨ 모든 작업 종료.")
if processed_results:
    print(f"최종 파일은 '{output_file}'에 저장되어 있습니다.")

## 중간에 멈췄다면 멈춘 부분부터 실행할 수 있도록 하기

In [ ]:

import sys
import types
from unittest.mock import MagicMock
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
import nest_asyncio
import time

# 1. 기존에 잘못 설정된 가짜 모듈이 있다면 제거 (청소)
if "google.generativeai" in sys.modules:
    del sys.modules["google.generativeai"]
if "google.ai.generativelanguage" in sys.modules:
    del sys.modules["google.ai.generativelanguage"]

# 2. 'google.generativeai'를 위한 정교한 가짜 모듈 생성
# 단순히 MagicMock()만 쓰면 __spec__ 에러가 나므로, 실제 모듈 객체(ModuleType)를 사용
mock_genai = types.ModuleType("google.generativeai")
mock_genai.__spec__ = MagicMock()       # __spec__ 속성 설정 (에러 방지 핵심!)
mock_genai.__path__ = []                # 패키지처럼 보이게 설정

# 3. sys.modules에 주입 (이제 파이썬은 이 모듈이 정상적으로 설치됐다고 착각합니다)
sys.modules["google.generativeai"] = mock_genai
sys.modules["google.ai.generativelanguage"] = MagicMock()

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import nest_asyncio
import os
from dotenv import load_dotenv
import logging


# 1. 환경 설정
nest_asyncio.apply()
# 경고 메시지 숨기기
logging.getLogger("ragas").setLevel(logging.ERROR)
logging.getLogger("langchain").setLevel(logging.ERROR)

model = ChatUpstage(model="solar-pro", temperature=0.5)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
#model = ChatOpenAI(model="gpt-4o")
#embeddings = OpenAIEmbeddings()

# 2. 파일 로드
file_summary = "GQ_3answers_2175.xlsx"
file_original = "GotQuestions_raw_Ara_2025-05-01.xlsx"
output_file = "[Ver2]GQ_Ragas_Progress_upstage.xlsx" # 중간 저장 파일명

print(f"📂 원본 데이터 로딩 중...")
df_summary = pd.read_excel(file_summary)
df_original = pd.read_excel(file_original)

# 데이터 병합 (이전과 동일)
def get_question_col(df):
    for col in ['Question', 'Question_ENG']:
        if col in df.columns:
            return col
    return df.columns[0]

q_col_sum = get_question_col(df_summary)
q_col_orig = get_question_col(df_original)

df_merged = pd.merge(
    df_summary, 
    df_original[[q_col_orig, 'Answer_ENG']], 
    left_on=q_col_sum, 
    right_on=q_col_orig, 
    how='left'
)

ground_truth_col = 'Answer_ENG'
final_q_col = q_col_sum
summary_cols = ['Summary1', 'Summary2', 'Summary3']

# 3. 이어하기 설정 (Resume Logic)
processed_results = []
start_idx_resume = 0

if os.path.exists(output_file):
    print(f"🔄 기존 진행 파일 발견: {output_file}")
    df_existing = pd.read_excel(output_file)
    start_idx_resume = len(df_existing) # 이미 처리된 행의 개수
    processed_results.append(df_existing) # 기존 결과 메모리에 로드
    print(f"   ▶ {start_idx_resume}개 데이터가 이미 처리되었습니다. {start_idx_resume}번부터 이어서 시작합니다.")
else:
    print("   ▶ 기존 파일이 없습니다. 처음(0번)부터 시작합니다.")

# 4. 배치 처리 설정
BATCH_SIZE = 10
total_rows = len(df_merged)
W_REL, W_COR, W_SIM = 0.5, 0.22, 0.28 # 가중치

def pick_best_weighted(row):
    scores = []
    answers = []
    for i in range(1, 4):
        s_rel = row.get(f'Summary{i}_Relevancy', 0)
        s_cor = row.get(f'Summary{i}_Correctness', 0)
        s_sim = row.get(f'Summary{i}_Similarity', 0)
        if pd.isna(s_rel): s_rel = 0
        if pd.isna(s_cor): s_cor = 0
        if pd.isna(s_sim): s_sim = 0
        
        weighted_score = (s_rel * W_REL) + (s_cor * W_COR) + (s_sim * W_SIM)
        scores.append(weighted_score)
        answers.append(row.get(f'Summary{i}', ""))
    
    max_score = max(scores)
    max_index = scores.index(max_score)
    return pd.Series({
        'Best_Summary': answers[max_index], 
        'Weighted_Score': max_score, 
        'Best_Source_Num': f"Summary {max_index+1}"
    })

# 5. 메인 루프 (중단된 지점부터 시작)
# range 시작점을 start_idx_resume로 설정하여 건너뛰기 구현
# 주의: BATCH_SIZE 단위로 딱 떨어지지 않을 수 있으므로 조정
current_start = (start_idx_resume // BATCH_SIZE) * BATCH_SIZE 
if start_idx_resume % BATCH_SIZE != 0:
    # 혹시 중간에 애매하게 끊겼다면, 안전하게 그 배치 처음부터 다시 하도록 설정
    print(f"   ⚠️ 배치가 중간에 끊겼습니다. {current_start}번부터 다시 처리합니다.")
else:
    current_start = start_idx_resume

for start_idx in range(current_start, total_rows, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_rows)
    print(f"\n🔄 Processing Batch: {start_idx} ~ {end_idx} (Total: {total_rows})")
    
    batch_df = df_merged.iloc[start_idx:end_idx].copy()
    
    questions = batch_df[final_q_col].fillna("").astype(str).tolist()
    ground_truths = batch_df[ground_truth_col].fillna("").astype(str).tolist()
    contexts = [[gt] for gt in ground_truths]
    
    current_batch_result = batch_df.copy()
    
    try:
        for i, col in enumerate(summary_cols):
            num = i + 1
            answers = batch_df[col].fillna("").astype(str).tolist()
            
            data_dict = {
                "question": questions,
                "ground_truth": ground_truths,
                "answer": answers,
                "contexts": contexts
            }
            dataset = Dataset.from_dict(data_dict)
            
            print(f"   ...Evaluating {col}")
            results = evaluate(
                dataset=dataset,
                metrics=[answer_correctness, answer_similarity, answer_relevancy],
                llm=model,
                embeddings=embeddings,
                raise_exceptions=False
            )
            res_df = results.to_pandas()
            current_batch_result[f'Summary{num}_Correctness'] = res_df['answer_correctness'].values
            current_batch_result[f'Summary{num}_Similarity'] = res_df['answer_similarity'].values
            current_batch_result[f'Summary{num}_Relevancy'] = res_df['answer_relevancy'].values

        best_pick_df = current_batch_result.apply(pick_best_weighted, axis=1)
        current_batch_result = pd.concat([current_batch_result, best_pick_df], axis=1)
        
        # 결과 리스트에 추가 (주의: 이미 있는 데이터면 덮어쓰거나 건너뛰어야 함)
        # 여기서는 매번 전체를 다시 concat해서 저장하는 방식 (가장 안전)
        if len(processed_results) > 0 and start_idx < len(pd.concat(processed_results)):
             # 만약 재시작으로 인해 겹치는 부분이 있다면, 기존 리스트에서 该 부분을 제거하고 새거 추가
             # 복잡함을 피하기 위해, 그냥 기존 파일 읽은거 + 새로 한거 합치기
             pass
        else:
             pass 

        # *** 중요: 이어쓰기 저장 로직 ***
        # 매 배치마다 파일 끝에 추가(append)하는 것이 효율적이지만, 엑셀은 append가 어려움
        # 따라서, '기존에 읽어온 df_existing'이 있다면 그것과 '새로 한 batch'를 합쳐서 저장
        
        if 'df_existing' in locals() and start_idx == current_start:
             # 첫 루프에서는 기존 것과 합치지 않고, processed_results를 초기화
             # 위에서 이미 append 했으므로 pass
             pass
        else:
             processed_results.append(current_batch_result)

        # 전체 병합 및 저장
        all_results_so_far = pd.concat(processed_results, axis=0)
        # 중복 제거 (혹시 재시작 시점에 겹친 행이 있다면 제거)
        all_results_so_far = all_results_so_far.drop_duplicates(subset=[final_q_col], keep='last')
        
        # 컬럼 순서 정리
        cols_order = [final_q_col, ground_truth_col] + summary_cols
        metrics_cols = []
        for i in range(1, 4):
            metrics_cols.extend([f'Summary{i}_Correctness', f'Summary{i}_Similarity', f'Summary{i}_Relevancy'])
        final_cols = cols_order + metrics_cols + ['Best_Source_Num', 'Weighted_Score', 'Best_Summary']
        existing_cols = [c for c in final_cols if c in all_results_so_far.columns]
        remaining_cols = [c for c in all_results_so_far.columns if c not in existing_cols]
        
        save_df = all_results_so_far[existing_cols + remaining_cols]
        save_df.to_excel(output_file, index=False)
        
        print(f"   💾 저장 완료: {len(save_df)}행 저장됨.")
        
        print("   💤 5초 대기...")
        time.sleep(5)

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        break

print("✨ 작업 종료.")

# 빈 값 채우기

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
from langchain_upstage import ChatUpstage, UpstageEmbeddings
import nest_asyncio
import os
from langchain_core.outputs import ChatResult

# 1. 환경 및 모델 설정
nest_asyncio.apply()

class SafeChatUpstage(ChatUpstage):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        if 'n' in kwargs: kwargs['n'] = 1
        try:
            return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)
        except Exception as e:
            if "n must be 1" in str(e):
                kwargs['n'] = 1
                return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)
            raise e

    async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        if 'n' in kwargs: kwargs['n'] = 1
        try:
            return await super()._agenerate(messages, stop=stop, run_manager=run_manager, **kwargs)
        except Exception as e:
            if "n must be 1" in str(e):
                kwargs['n'] = 1
                return await super()._agenerate(messages, stop=stop, run_manager=run_manager, **kwargs)
            raise e

# 타임아웃 2분 설정
model = SafeChatUpstage(model="solar-pro", temperature=0.0, timeout=120, max_retries=3)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large", timeout=120)

# =================================================================
# 2. 파일 경로 설정
# =================================================================
#summary_file = "GQ_Ragas_Final_Validated_2175.xlsx"    # [대상] 텍스트가 완벽한 파일 (점수 구멍 있음)
summary_file = "GQ_Ragas_Final_Repaired.xlsx"    # [대상] 텍스트가 완벽한 파일 (점수 구멍 있음)
raw_file = "GotQuestions_raw_Ara_2025-05-01.xlsx"      # [정답] 원본 긴 답변이 있는 파일
final_output_file = "GQ_Ragas_Final_Repaired2.xlsx"      # [결과] 최종 저장 파일

print("🚀 최종 점수 채우기 작업을 시작합니다...")


if os.path.exists(final_output_file):
    print(f"🔄 작업 중이던 파일 발견! ({final_output_file}) 로드 중...")
    df_main = pd.read_excel(final_output_file)
else:
    print("✨ 새로 병합하여 작업을 시작합니다...")
    df_sum = pd.read_excel(summary_file)
    df_raw = pd.read_excel(raw_file)
    
    print(f"   - 요약 파일: {len(df_sum)}행")
    print(f"   - 원본 파일: {len(df_raw)}행, 컬럼: {df_raw.columns.tolist()}") # 컬럼 확인용 출력 추가

    # [안전 장치 1] 원본 파일에서 정답 컬럼 이름 찾기
    tgt_ans_col = 'Answer_ENG'
    if tgt_ans_col not in df_raw.columns:
        # 혹시 다른 이름인지 확인
        possible_names = ['Answer', 'answer', 'Answer_Kor', 'English Answer']
        for name in possible_names:
            if name in df_raw.columns:
                tgt_ans_col = name
                break
        print(f"   ℹ️ 원본 정답 컬럼 확인됨: '{tgt_ans_col}'")

    # 3-1. 원본 중복 제거
    df_raw_dedup = df_raw.drop_duplicates(subset=['Question_ENG'], keep='first')

    # [안전 장치 2] 병합 전, 가져올 컬럼만 명확히 선택하고 이름을 미리 변경 (충돌 방지)
    # 원본에서 'Question_ENG'와 '정답 컬럼'만 가져오되, 정답 컬럼 이름을 'New_Ground_Truth'로 변경해서 가져옵니다.
    df_raw_slice = df_raw_dedup[['Question_ENG', tgt_ans_col]].rename(columns={tgt_ans_col: 'New_Ground_Truth'})

    # 3-2. 병합 (요약 파일 기준)
    df_main = pd.merge(
        df_sum, 
        df_raw_slice, 
        left_on='Question', 
        right_on='Question_ENG', 
        how='left'
    )
    
    # 정답 컬럼 설정 (이제 이름 충돌 걱정 없음)
    df_main['Final_Ground_Truth'] = df_main['New_Ground_Truth']
    
    # 필요 없다면 임시 컬럼 삭제
    # df_main = df_main.drop(columns=['New_Ground_Truth']) 

    # 병합 후 검증
    if len(df_main) != len(df_sum):
        print(f"❌ 경고: 병합 후 행 개수가 다릅니다! ({len(df_main)}행)")
        df_main = df_main.drop_duplicates(subset=['Question'])
    else:
        print(f"   ✅ 병합 완료. 행 개수 정상 ({len(df_main)}행)")

    df_main.to_excel(final_output_file, index=False)


# 4. 점수가 0이거나 빈 행 찾기 (계산 대상)
def get_todo_indices(df):
    indices = []
    # Summary1의 Correctness가 0이면 아직 계산 안 된 것으로 간주
    for idx, row in df.iterrows():
        val = row.get('Summary1_Correctness', 0)
        if pd.isna(val) or val == 0.0:
            indices.append(idx)
    return indices

todo_indices = get_todo_indices(df_main)
print(f"🚨 계산이 필요한 행: {len(todo_indices)}개 / 전체 {len(df_main)}개")

# 5. 평가 및 저장 루프
BATCH_SIZE = 2
summary_cols = ['Summary1', 'Summary2', 'Summary3']
W_REL, W_COR, W_SIM = 0.5, 0.22, 0.28

if len(todo_indices) > 0:
    print("🚀 평가 시작 (2개씩 저장)...")
    
    for i in range(0, len(todo_indices), BATCH_SIZE):
        batch_idxs = todo_indices[i : i + BATCH_SIZE]
        print(f"\n🔄 Processing indices: {batch_idxs} ({i+1}/{len(todo_indices)})")
        
        # 현재 배치 데이터 준비
        batch_df = df_main.loc[batch_idxs].copy()
        
        questions = batch_df['Question'].astype(str).tolist()
        ground_truths = batch_df['Final_Ground_Truth'].astype(str).tolist()
        contexts = [[gt] if isinstance(gt, str) and len(gt) > 0 else [" "] for gt in ground_truths]
        
        try:
            # Summary 1, 2, 3 각각 평가
            for s_idx, col in enumerate(summary_cols):
                num = s_idx + 1
                answers = batch_df[col].fillna("").astype(str).tolist()
                
                # 내용이 없으면 스킵
                if not any(answers): continue

                data_dict = {
                    "question": questions,
                    "ground_truth": ground_truths,
                    "answer": answers,
                    "contexts": contexts
                }
                dataset = Dataset.from_dict(data_dict)
                
                # Ragas 평가 실행
                results = evaluate(
                    dataset=dataset,
                    metrics=[answer_correctness, answer_similarity, answer_relevancy],
                    llm=model,
                    embeddings=embeddings,
                    raise_exceptions=False
                )
                
                res_df = results.to_pandas()
                
                # 결과 df_main에 업데이트
                if not res_df.empty:
                    df_main.loc[batch_idxs, f'Summary{num}_Correctness'] = res_df['answer_correctness'].values
                    df_main.loc[batch_idxs, f'Summary{num}_Similarity'] = res_df['answer_similarity'].values
                    df_main.loc[batch_idxs, f'Summary{num}_Relevancy'] = res_df['answer_relevancy'].values

            # Best Pick(가중치 점수) 재계산
            def pick_best(row):
                scores = []
                for k in range(1, 4):
                    s_rel = row.get(f'Summary{k}_Relevancy', 0) or 0
                    s_cor = row.get(f'Summary{k}_Correctness', 0) or 0
                    s_sim = row.get(f'Summary{k}_Similarity', 0) or 0
                    
                    score = (s_rel * W_REL) + (s_cor * W_COR) + (s_sim * W_SIM)
                    scores.append(score)
                
                best_idx = scores.index(max(scores))
                return pd.Series({
                    'Best_Summary': row.get(f'Summary{best_idx+1}', ""),
                    'Weighted_Score': max(scores),
                    'Best_Source_Num': f"Summary {best_idx+1}"
                })

            updated_best = df_main.loc[batch_idxs].apply(pick_best, axis=1)
            df_main.loc[batch_idxs, ['Best_Summary', 'Weighted_Score', 'Best_Source_Num']] = updated_best[['Best_Summary', 'Weighted_Score', 'Best_Source_Num']]
            
            # ★ 2개 처리할 때마다 즉시 저장
            df_main.to_excel(final_output_file, index=False)
            print(f"   ✅ {final_output_file} 저장 완료")

        except Exception as e:
            print(f"❌ 에러 발생 (해당 배치는 건너뜀): {e}")
            continue

    print(f"\n✨ 모든 작업 완료! 최종 파일: {final_output_file}")
else:
    print("✨ 모든 데이터가 이미 처리되어 있습니다.")

# 파일이 동일한지 검사

In [ ]:
######두 파일이 동일한지 검사###### question, summary123

import pandas as pd

# 1. 파일명 설정
file1 = "[원본]GQ_3answers_2175.xlsx"          # 원본 요약 파일
file2 = "GQ_Ragas_Final_Repaired_Revised.xlsx"    # 정리된 최종 파일

print("📂 파일 로딩 및 검증 준비...")
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)

# 2. 비교할 4개 컬럼 지정
cols_to_check = ['Question', 'Summary1', 'Summary2', 'Summary3']

# [방어 코드] 컬럼 존재 여부 확인
for df, fname in [(df1, file1), (df2, file2)]:
    missing = [c for c in cols_to_check if c not in df.columns]
    if missing:
        # 혹시 Question_ENG 같은 이름일 경우를 대비해 알림
        print(f"❌ {fname} 파일에서 컬럼을 찾을 수 없습니다: {missing}")
        print(f"   (현재 컬럼 목록: {df.columns.tolist()})")
        # 멈추지 않고 넘어가려면 이 부분을 수정해야 하지만, 정확한 비교를 위해선 필수입니다.
        exit()

print(f"   - 파일1 행 개수: {len(df1)}")
print(f"   - 파일2 행 개수: {len(df2)}")

# 3. 데이터 전처리 (공백 제거 및 문자열 변환)
# 미세한 띄어쓰기 차이나 NaN(빈칸)으로 인한 불일치를 막기 위해 필수입니다.
def clean_data(df, cols):
    df_subset = df[cols].copy()
    for col in cols:
        # 문자열로 변환 -> 앞뒤 공백 제거 -> 빈칸은 ""로 통일
        df_subset[col] = df_subset[col].astype(str).str.strip().fillna("")
    return df_subset

df1_clean = clean_data(df1, cols_to_check)
df2_clean = clean_data(df2, cols_to_check)

# 4. [핵심] 병합(Merge)을 이용한 교집합/차집합 확인
# indicator=True 옵션을 쓰면 데이터가 어느 쪽에만 있는지(_merge 컬럼) 알려줍니다.
merged = pd.merge(
    df1_clean,
    df2_clean,
    on=cols_to_check,
    how='outer', # 양쪽 다 확인
    indicator=True
)

# 5. 결과 분석
only_in_1 = merged[merged['_merge'] == 'left_only']
only_in_2 = merged[merged['_merge'] == 'right_only']
both = merged[merged['_merge'] == 'both']

print("\n" + "="*50)
print("📊 검증 결과 리포트")
print("="*50)

print(f"✅ 두 파일에 모두 존재하는 행 (일치): {len(both)}개")

if len(only_in_1) == 0 and len(only_in_2) == 0:
    print("\n🎉 완벽합니다! 두 파일의 내용(4개 열)이 100% 일치합니다.")
else:
    print("\n⚠️ 불일치 발생!")
    
    if len(only_in_1) > 0:
        print(f"\n📍 [파일 1에만 있음] (파일 2에 누락됨): {len(only_in_1)}개")
        print("   -> 예시 (상위 3개):")
        print(only_in_1['Question'].head(3).to_string(index=False))

    if len(only_in_2) > 0:
        print(f"\n📍 [파일 2에만 있음] (파일 1에 없음): {len(only_in_2)}개")
        print("   -> 예시 (상위 3개):")
        print(only_in_2['Question'].head(3).to_string(index=False))

print("-" * 50)

# 0이나 공백 포함된 행 다시 Ragas 평가 

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, answer_similarity, answer_relevancy
from langchain_upstage import ChatUpstage, UpstageEmbeddings
import nest_asyncio
import os
from langchain_core.outputs import ChatResult

# 1. 환경 및 모델 설정
nest_asyncio.apply()

class SafeChatUpstage(ChatUpstage):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        if 'n' in kwargs: kwargs['n'] = 1
        try:
            return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)
        except Exception as e:
            if "n must be 1" in str(e):
                kwargs['n'] = 1
                return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)
            raise e

    async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        if 'n' in kwargs: kwargs['n'] = 1
        try:
            return await super()._agenerate(messages, stop=stop, run_manager=run_manager, **kwargs)
        except Exception as e:
            if "n must be 1" in str(e):
                kwargs['n'] = 1
                return await super()._agenerate(messages, stop=stop, run_manager=run_manager, **kwargs)
            raise e

# 타임아웃 2분 설정
model = SafeChatUpstage(model="solar-pro", temperature=0.0, timeout=120, max_retries=3)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large", timeout=120)

# =================================================================
# 2. 파일 경로 설정
# =================================================================
summary_file = "GQ_Ragas_Final_Repaired2.xlsx"    # [대상] 텍스트가 완벽한 파일
raw_file = "GotQuestions_raw_Ara_2025-05-01.xlsx"      # [정답] 원본 긴 답변이 있는 파일
final_output_file = "GQ_Ragas_Final_Repaired_Revised.xlsx"      # [결과] 최종 저장 파일

print("🚀 최종 점수 채우기 작업을 시작합니다...")

if os.path.exists(final_output_file):
    print(f"🔄 작업 중이던 파일 발견! ({final_output_file}) 로드 중...")
    df_main = pd.read_excel(final_output_file)
else:
    print("✨ 새로 병합하여 작업을 시작합니다...")
    df_sum = pd.read_excel(summary_file)
    df_raw = pd.read_excel(raw_file)
    
    # 중복 제거 및 병합 준비
    df_raw_dedup = df_raw.drop_duplicates(subset=['Question_ENG'], keep='first')
    tgt_ans_col = 'Answer_ENG' 
    # 컬럼명이 다를 경우를 대비한 체크
    if tgt_ans_col not in df_raw.columns:
        for name in ['Answer', 'answer', 'Answer_Kor', 'English Answer']:
            if name in df_raw.columns:
                tgt_ans_col = name
                break
    
    df_raw_slice = df_raw_dedup[['Question_ENG', tgt_ans_col]].rename(columns={tgt_ans_col: 'Ground_Truth'})

    # 병합
    df_main = pd.merge(
        df_sum, 
        df_raw_slice, 
        left_on='Question', 
        right_on='Question_ENG', 
        how='left'
    )
    df_main['Final_Ground_Truth'] = df_main['Ground_Truth']
    
    if len(df_main) != len(df_sum):
        df_main = df_main.drop_duplicates(subset=['Question'])

    df_main.to_excel(final_output_file, index=False)


# =================================================================
# 4. 점수가 0이거나 빈 행 찾기 (수정된 부분)
# =================================================================
def get_todo_indices(df):
    indices = []
    
    # 검사할 모든 지표 컬럼
    target_cols = [
        'Summary1_Correctness', 'Summary1_Similarity', 'Summary1_Relevancy',
        'Summary2_Correctness', 'Summary2_Similarity', 'Summary2_Relevancy',
        'Summary3_Correctness', 'Summary3_Similarity', 'Summary3_Relevancy'
    ]

    for idx, row in df.iterrows():
        is_bad_row = False
        
        # 각 컬럼을 순회하며 '나쁜 값'이 있는지 확인
        for col in target_cols:
            val = row.get(col, None) # 컬럼이 없으면 None 반환 -> 불량 처리됨
            
            # 1. 값이 없거나(NaN/None)
            if pd.isna(val):
                is_bad_row = True
                break
            
            # 2. 숫자 0 인 경우 (0.0 포함)
            if isinstance(val, (int, float)) and val == 0:
                is_bad_row = True
                break
                
            # 3. 문자열인데 공백이거나 '0'인 경우
            s_val = str(val).strip()
            if s_val == '' or s_val == '0' or s_val == '0.0':
                is_bad_row = True
                break
        
        if is_bad_row:
            indices.append(idx)
            
    return indices

todo_indices = get_todo_indices(df_main)
print(f"🚨 재계산이 필요한 행: {len(todo_indices)}개 / 전체 {len(df_main)}개")

# =================================================================
# 5. 평가 및 저장 루프
# =================================================================
BATCH_SIZE = 2
summary_cols = ['Summary1', 'Summary2', 'Summary3']
W_REL, W_COR, W_SIM = 0.5, 0.22, 0.28

if len(todo_indices) > 0:
    print("🚀 평가 시작 (2개씩 저장)...")
    
    for i in range(0, len(todo_indices), BATCH_SIZE):
        batch_idxs = todo_indices[i : i + BATCH_SIZE]
        print(f"\n🔄 Processing indices: {batch_idxs} ({i+1}/{len(todo_indices)})")
        
        # 현재 배치 데이터 준비
        batch_df = df_main.loc[batch_idxs].copy()
        
        questions = batch_df['Question'].astype(str).tolist()
        ground_truths = batch_df['Final_Ground_Truth'].astype(str).tolist()
        # Contexts가 비어있으면 에러가 날 수 있으므로 최소한의 공백 리스트 처리
        contexts = [[gt] if isinstance(gt, str) and len(gt) > 0 else [" "] for gt in ground_truths]
        
        try:
            # Summary 1, 2, 3 각각 평가
            for s_idx, col in enumerate(summary_cols):
                num = s_idx + 1
                answers = batch_df[col].fillna("").astype(str).tolist()
                
                # 내용이 없으면 스킵 (하지만 점수는 채워야 하니 사실상 텍스트가 있어야 함)
                if not any(answers): 
                    print(f"   ⚠️ {col} 텍스트가 모두 비어있습니다. 건너뜁니다.")
                    continue

                data_dict = {
                    "question": questions,
                    "ground_truth": ground_truths,
                    "answer": answers,
                    "contexts": contexts
                }
                dataset = Dataset.from_dict(data_dict)
                
                # Ragas 평가 실행
                results = evaluate(
                    dataset=dataset,
                    metrics=[answer_correctness, answer_similarity, answer_relevancy],
                    llm=model,
                    embeddings=embeddings,
                    raise_exceptions=False
                )
                
                res_df = results.to_pandas()
                
                # 결과 df_main에 업데이트
                if not res_df.empty:
                    df_main.loc[batch_idxs, f'Summary{num}_Correctness'] = res_df['answer_correctness'].values
                    df_main.loc[batch_idxs, f'Summary{num}_Similarity'] = res_df['answer_similarity'].values
                    df_main.loc[batch_idxs, f'Summary{num}_Relevancy'] = res_df['answer_relevancy'].values

            # Best Pick(가중치 점수) 재계산
            def pick_best(row):
                scores = []
                for k in range(1, 4):
                    s_rel = row.get(f'Summary{k}_Relevancy', 0) or 0
                    s_cor = row.get(f'Summary{k}_Correctness', 0) or 0
                    s_sim = row.get(f'Summary{k}_Similarity', 0) or 0
                    
                    score = (s_rel * W_REL) + (s_cor * W_COR) + (s_sim * W_SIM)
                    scores.append(score)
                
                best_idx = scores.index(max(scores))
                return pd.Series({
                    'Best_Summary': row.get(f'Summary{best_idx+1}', ""),
                    'Weighted_Score': max(scores),
                    'Best_Source_Num': f"Summary {best_idx+1}"
                })

            updated_best = df_main.loc[batch_idxs].apply(pick_best, axis=1)
            df_main.loc[batch_idxs, ['Best_Summary', 'Weighted_Score', 'Best_Source_Num']] = updated_best[['Best_Summary', 'Weighted_Score', 'Best_Source_Num']]
            
            # 저장
            df_main.to_excel(final_output_file, index=False)
            print(f"   ✅ {final_output_file} 저장 완료")

        except Exception as e:
            print(f"❌ 에러 발생 (해당 배치는 건너뜀): {e}")
            continue

    print(f"\n✨ 모든 작업 완료! 최종 파일: {final_output_file}")
else:
    print("✨ 모든 데이터가 이미 정상적으로 처리되어 있습니다 (0, NaN, 공백 없음).")